# Défi — Système RAG avec LangChain & Hugging Face

Système de **génération augmentée par récupération (RAG)** capable de répondre à des questions
à partir du jeu de données `databricks/databricks-dolly-15k` :
- **LangChain** orchestre les composants
- **`HuggingFaceDatasetLoader`** charge les données
- **`RecursiveCharacterTextSplitter`** découpe les documents
- **`all-MiniLM-L6-v2`** génère les embeddings
- **FAISS** indexe et récupère
- **`Intel/dynamic_tinybert`** (modèle de question-answering) génère les réponses
- **`RetrievalQA`** relie récupération et réponse

À exécuter de préférence sur **Google Colab**.

## 1. Configuration de l'environnement

On épingle **LangChain 0.3.x** : c'est la version où `HuggingFaceDatasetLoader`
(`langchain.document_loaders`) et `RetrievalQA` (`langchain.chains`) existent tels que
l'exercice les utilise. LangChain 1.x a déplacé/retiré ces composants.

💡 Si Colab affiche un avertissement de dépendances, faites *Exécution → Redémarrer la session*
puis relancez à partir des imports.

In [ ]:
%%capture
!pip install -q "langchain==0.3.27"
!pip install -q "langchain-community==0.3.27"
!pip install -q "langchain-huggingface==0.3.1"
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q "datasets<3"
!pip install -q faiss-cpu

## 2. Charger le jeu de données

`HuggingFaceDatasetLoader` récupère un dataset Hugging Face et le formate en `Document` LangChain.
On utilise la colonne **`context`** comme contenu principal.

In [ ]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader

dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

print("Nombre de documents chargés :", len(data))
print(data[:2])   # aperçu des 2 premières entrées

> ℹ️ Beaucoup d'entrées de Dolly ont un `context` vide (questions ouvertes sans contexte).
> Pour un RAG plus pertinent, on peut filtrer les documents vides — voir la cellule suivante
> (facultatif mais recommandé).

In [ ]:
# (Recommandé) Retirer les documents dont le contenu est vide
data = [d for d in data if d.page_content and d.page_content.strip()]
print("Documents non vides :", len(data))

## 3. Découper les documents en chunks

`RecursiveCharacterTextSplitter` avec `chunk_size=1000` et `chunk_overlap=150`.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)

print("Nombre de chunks :", len(docs))
print(docs[0])

## 4. Intégrer le texte (embeddings)

Modèle `sentence-transformers/all-MiniLM-L6-v2` via `HuggingFaceEmbeddings`.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

modelPath = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

In [ ]:
# (Facultatif) Test rapide de l'embedding
text = "This is a test document."
query_result = embeddings.embed_query(text)
print("Dimension de l'embedding :", len(query_result))
print("3 premières valeurs :", query_result[:3])

## 5. Créer le vector store FAISS

⚠️ Sur les 15k exemples, l'embedding peut être long. Pour aller vite, on peut se limiter à un
sous-ensemble (`docs[:2000]`). Retirez la découpe pour indexer tout le dataset.

In [ ]:
from langchain_community.vectorstores import FAISS

# Sous-ensemble pour accélérer (ajustez ou supprimez [:2000] pour tout indexer)
docs_to_index = docs[:2000]

db = FAISS.from_documents(docs_to_index, embeddings)
print("Vector store FAISS créé avec", db.index.ntotal, "vecteurs")

## 6. Préparer le modèle LLM (question-answering)

On charge `Intel/dynamic_tinybert`, un modèle **extractif** de question-answering, et on crée un
pipeline `question-answering`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline

model_name = "Intel/dynamic_tinybert"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True, max_length=512)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

question_answerer = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
)
print("Pipeline de question-answering prêt :", model_name)

### ⚠️ Note technique importante

`Intel/dynamic_tinybert` est un modèle **extractif** (`AutoModelForQuestionAnswering`) : il
extrait un empan de réponse à partir d'un couple *(question, contexte)*. Il **n'est pas** un
modèle génératif. Or `langchain.HuggingFacePipeline` + `RetrievalQA` attendent un pipeline de
**génération** (`text-generation` / `text2text-generation`). Enrober un pipeline `question-answering`
dans `HuggingFacePipeline` puis appeler `RetrievalQA` lève une erreur au moment de la requête.

Deux solutions, présentées ci-dessous :
- **6A (recommandée, fonctionne)** — utiliser le pipeline extractif tel qu'il est prévu :
  récupérer le contexte via le retriever FAISS, puis extraire la réponse.
- **6B (variante génénrative)** — remplacer le modèle par un LLM génératif (`flan-t5`) pour
  utiliser la vraie chaîne `RetrievalQA` comme dans l'énoncé.

## 7. Chaîne de questions-réponses par récupération

On crée un retriever à partir de FAISS (`k=4`).

In [ ]:
retriever = db.as_retriever(search_kwargs={"k": 4})
print("Retriever prêt (k=4)")

### 7A. Réponse extractive (recommandée avec `dynamic_tinybert`)

On combine le retriever et le pipeline extractif : le retriever fournit le contexte, le modèle
extrait la réponse.

In [ ]:
def rag_extractive_answer(question, retriever, qa_pipeline, k=4):
    # 1) Récupérer les documents pertinents
    retrieved_docs = retriever.invoke(question)
    # 2) Concaténer leur contenu comme contexte
    context = " ".join(d.page_content for d in retrieved_docs)
    # 3) Extraire la réponse
    result = qa_pipeline(question=question, context=context)
    return result, retrieved_docs

### 7B. (Variante) Vraie chaîne `RetrievalQA` avec un LLM génératif

Cette version suit l'énoncé (`RetrievalQA.from_chain_type(..., chain_type="refine")`) mais avec un
modèle **génératif** compatible. Décommentez pour l'utiliser.

In [ ]:
# from transformers import AutoTokenizer as _Tok, AutoModelForSeq2SeqLM, pipeline as _pipe
# from langchain_huggingface import HuggingFacePipeline
# from langchain.chains import RetrievalQA
#
# gen_id = "google/flan-t5-base"
# gen_tok = _Tok.from_pretrained(gen_id)
# gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_id)
# gen_pipe = _pipe("text2text-generation", model=gen_model, tokenizer=gen_tok, max_new_tokens=256)
# llm = HuggingFacePipeline(pipeline=gen_pipe)
#
# qa = RetrievalQA.from_chain_type(
#     llm=llm, chain_type="refine", retriever=retriever, return_source_documents=False
# )
# print(qa.invoke({"query": "What is cheesemaking?"})["result"])

## 8. Tester le système RAG

In [ ]:
question = "What is cheesemaking?"

result, sources = rag_extractive_answer(question, retriever, question_answerer, k=4)

print("Question :", question)
print("Réponse  :", result["answer"])
print("Score    :", round(result["score"], 4))
print("\nExtraits sources utilisés :")
for i, d in enumerate(sources, 1):
    print(f"  {i}. {d.page_content[:120].strip()}...")

In [ ]:
# Essayez d'autres questions
for q in ["What is a computer?", "What is machine learning?", "Who wrote Hamlet?"]:
    res, _ = rag_extractive_answer(q, retriever, question_answerer, k=4)
    print(f"Q: {q}\n   -> {res['answer']} (score={res['score']:.3f})\n")

## ✅ Récapitulatif

Vous avez construit un pipeline RAG complet : chargement de `databricks-dolly-15k` via
`HuggingFaceDatasetLoader`, découpage en chunks (1000/150), embeddings `all-MiniLM-L6-v2`,
vector store **FAISS**, retriever (`k=4`), puis réponse aux questions.

**Point clé appris :** le choix du modèle doit correspondre à la tâche. `Intel/dynamic_tinybert`
est **extractif** (question + contexte → empan de réponse) et s'utilise via le pipeline
`question-answering`, tandis que la chaîne `RetrievalQA` de LangChain attend un modèle
**génératif** (`flan-t5`, GPT, …). Les deux approches sont fournies ci-dessus.